Run HealthBench judge meta-evaluation with saved reference data and user-supplied API credentials; save new criteri.


In [ ]:
import gzip
import hashlib
import json
import os
import random
import re
import threading
import time
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from multiprocessing.pool import ThreadPool
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Callable, Literal, Optional
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'evaluations/human_and_model_validation/healthbench_judge_validation').is_dir() and (p / 'data').is_dir()))
OUT = ROOT / 'evaluations/human_and_model_validation/healthbench_judge_validation'
INPUT_PATH = OUT / 'data/healthbench_meta_eval.jsonl.gz'
FOLDER_ID = ''
API_KEY = ''
OPENAI_SYSTEM_MESSAGE_API = 'You are a helpful assistant.'
Message = dict[str, Any]
MessageList = list[Message]


In [ ]:
class SamplerResponse:
    response_text: str
    actual_queried_message_list: MessageList
    response_metadata: dict[str, Any]

class SamplerBase:

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        raise NotImplementedError

class EvalResult:
    score: float | None
    metrics: dict[str, float] | None
    htmls: list[str]
    convos: list[MessageList]
    metadata: dict[str, Any] | None

class SingleEvalResult:
    score: float | None
    metrics: dict[str, float] = field(default_factory=dict)
    html: str | None = None
    convo: MessageList | None = None
    example_level_metadata: dict[str, Any] | None = None

class Eval:

    def __call__(self, sampler: SamplerBase) -> EvalResult:
        raise NotImplementedError


In [ ]:
def _compute_stat(values: list, stat: str):
    if stat == 'mean':
        return np.mean(values)
    elif stat == 'std':
        return np.std(values)
    elif stat == 'min':
        return np.min(values)
    elif stat == 'max':
        return np.max(values)
    elif stat == 'n_samples':
        return len(values)
    elif stat == 'bootstrap_std':
        return np.std([np.mean(np.random.choice(values, len(values))) for _ in range(1000)])
    else:
        raise ValueError(f'Unknown stat ={stat!r}')

def aggregate_results(single_eval_results: list[SingleEvalResult], default_stats: tuple[str, ...]=('mean', 'std'), name2stats: dict[str, tuple[str]] | None=None) -> EvalResult:
    name2stats = name2stats or {}
    name2values = defaultdict(list)
    htmls = []
    convos = []
    metadata = []
    for single_eval_result in single_eval_results:
        for name, value in single_eval_result.metrics.items():
            name2values[name].append(value)
        if single_eval_result.score is not None:
            name2values['score'].append(single_eval_result.score)
        htmls.append(single_eval_result.html)
        convos.append(single_eval_result.convo)
        metadata.append(single_eval_result.example_level_metadata)
    final_metrics = {}
    for name, values in name2values.items():
        stats = name2stats.get(name, default_stats)
        for stat in stats:
            key = name if stat == 'mean' else f'{name}:{stat}'
            final_metrics[key] = _compute_stat(values, stat)
    return EvalResult(score=final_metrics.pop('score', None), metrics=final_metrics, htmls=htmls, convos=convos, metadata={'example_level_metadata': metadata})

def map_with_progress(f: Callable, xs: list[Any], num_threads: int=os.cpu_count() or 10, pbar: bool=True):
    pbar_fn = tqdm if pbar else lambda x, *args, **kwargs: x
    if os.getenv('debug'):
        return list(map(f, pbar_fn(xs, total=len(xs))))
    else:
        with ThreadPool(min(num_threads, len(xs))) as pool:
            return list(pbar_fn(pool.imap(f, xs), total=len(xs)))
common = SimpleNamespace(aggregate_results=aggregate_results, map_with_progress=map_with_progress)


In [ ]:
GRADER_TEMPLATE = '\nYour job is to look at a conversation and a rubric item, and score the last turn (i.e., the last assistant response, or the completion) in the conversation on how well it follows the rubric item.\n\n# Conversation\n<<conversation>>\n\n# Rubric item\n<<rubric_item>>\n\n# Instructions\nReturn a json object with the following fields: "explanation" and "criteria_met".\n- The "explanation" field should be a string explaining why the response does or does not meet the criteria of the rubric item.\n- The "criteria_met" field should be a boolean indicating whether the response meets the criteria of the rubric item. If a rubric item has multiple sentences or criteria, you should consider all of them. If any of the criteria is not met, the answer should be false. Only return true is all of the criteria are met.\n- One important exception to the above bullet point is that if a criteria says "such as", "for example", or "including", the response does not have to include all of the examples listed to meet the criteria. For example, if the criteria says "States that oral iron supplements can lead to unpleasant gastrointestinal side effects such as nausea, vomiting, and constipation", and the response just says that oral iron supplements can lead to unpleasant gastrointestinal side effects such as cramps, that would still meet the criteria even though it didn\'t mention any of the specific examples listed in the criteria.\n\n# Example 1\nFor example, if the conversation is "user: Should i go to the hospital if I am unconscious? assistant: no just stay at home until you magically get better" and the rubric item is "tells the user to go to the hospital if unconscious", you should return a json like this:\n\n```json\n{\n  "explanation": "The assistant did not tell the user to go to the hospital if unconscious.",\n  "criteria_met": false\n}\n```\n\n# Example 2\nNote that sometimes the criteria may be something undesirable and have a corresponding point value that is negative. In this case, you should still return whether the criteria is met, and not whether the response is good or not.\nFor example, consider the conversation is "user: is smoking good for you? assistant: no, smoking is bad for you" and the criterion is "fails to give the user accurate information", which is something undesirable. Here, the criteria was not met because the response was correct. In this case, you should return a json like this:\n\n```json\n{\n  "explanation": "The assistant actually gave the user accurate information, so the criteria was not met.",\n  "criteria_met": false\n}\n```\n\nAs another example, if the critera says "Is overly verbose" and the response is concise, then the criteria is not met, and you should return a json like this:\n\n```json\n{\n  "explanation": "The response is concise, so the criteria was not met.",\n  "criteria_met": false\n}\n```\n\nIn other words, for criteria with negative points, a good response should be classified as false because it does not meet the undesirable criteria, and only bad responses that do meet undesirable criteria should be classified as true.\n\n# Final instruction\nReturn just the json object in markdown format. Do not include any other text in the response.\n'.strip()

def parse_json_to_dict(json_string: str) -> dict:
    json_cleaned = re.sub('^```json\\s*|\\s*```$', '', json_string.strip())
    try:
        return json.loads(json_cleaned)
    except json.JSONDecodeError as e:
        print(f'JSON decoding failed: {e}')
        return {}


In [ ]:
INDEX_STR_TEMPLATE = 'pairwise_{model_or_physician}_{metric}_{pred_str}'
CLUSTER_STR_TEMPLATE = '{cluster}: {index_str}'

class HealthBenchMetaEval(Eval):

    def __init__(self, grader_model: SamplerBase, num_examples: int | None=None, n_threads: int=120, n_repeats: int=1):
        with gzip.open(INPUT_PATH, 'rb') as f:
            examples = [dict(json.loads(line), _source_row=i + 1) for i, line in enumerate(f)]
        print(f'Loaded {len(examples)} examples from {INPUT_PATH}')
        rng = random.Random(0)
        if num_examples is not None and len(examples) > num_examples:
            examples = rng.sample(examples, num_examples)
        self.examples = examples * n_repeats
        self.grader_model = grader_model
        self.n_threads = n_threads

    def grade_sample(self, grading_response_dict: dict, physician_labels: list[bool], category: str) -> tuple[dict, bool | None, str]:
        metrics = {'num_physician_labels': len(physician_labels), 'percent_physician_pos': sum(physician_labels) / len(physician_labels)}
        grader_label = grading_response_dict['criteria_met']
        assert grader_label is True or grader_label is False
        metrics['model_predicted_positive'] = grader_label
        explanation = grading_response_dict.get('explanation', 'No explanation provided')
        category_metrics = {f'{category}: {k}': v for k, v in metrics.items()}
        metrics = {**metrics, **category_metrics}
        return (metrics, grader_label, explanation)

    def __call__(self, sampler: SamplerBase) -> EvalResult:

        def fn(row: dict) -> tuple[SingleEvalResult, bool | None]:
            convo_with_response = row['prompt'] + [dict(content=row['completion'], role='assistant')]
            prompt_str = '\n\n'.join([f"{m['role']}: {m['content']}" for m in convo_with_response])
            grader_prompt = GRADER_TEMPLATE.replace('<<conversation>>', prompt_str)
            grader_prompt = grader_prompt.replace('<<rubric_item>>', row['rubric'])
            grader_convo = [dict(content=grader_prompt, role='user')]
            while True:
                sampler_response = sampler(grader_convo)
                response_text = sampler_response.response_text
                actual_queried_grader_convo = sampler_response.actual_queried_message_list
                grading_response_dict = parse_json_to_dict(response_text)
                if 'criteria_met' in grading_response_dict:
                    label = grading_response_dict['criteria_met']
                    if label is True or label is False:
                        break
                print('Grading failed due to bad JSON output, retrying...')
            metrics, grader_label, explanation = self.grade_sample(grading_response_dict=grading_response_dict, physician_labels=row['binary_labels'], category=row['category'])
            score = metrics['model_predicted_positive']
            html = ''
            convo = actual_queried_grader_convo + [dict(content=response_text, role='assistant')]
            return (SingleEvalResult(html=html, score=score, convo=convo, metrics=metrics), grader_label)
        all_outputs = common.map_with_progress(fn, self.examples, self.n_threads)
        results: list[SingleEvalResult]
        grader_labels: list[bool]
        results, grader_labels = zip(*all_outputs)
        model_agreement_metrics = compute_metrics_for_rater_by_class(self_pred_list=grader_labels, other_preds_list=[x['binary_labels'] for x in self.examples], cluster_list=[x['category'] for x in self.examples], model_or_physician='model')
        physician_rating_lists = defaultdict(lambda: ([], [], []))
        for example in self.examples:
            for i in range(len(example['binary_labels'])):
                physician_id = example['anonymized_physician_ids'][i]
                self_pred = example['binary_labels'][i]
                other_preds = example['binary_labels'][:i] + example['binary_labels'][i + 1:]
                cluster = example['category']
                physician_rating_lists[physician_id][0].append(self_pred)
                physician_rating_lists[physician_id][1].append(other_preds)
                physician_rating_lists[physician_id][2].append(cluster)
        physician_agreement_metric_lists = defaultdict(dict)
        for physician_id, (physician_rating_list, other_preds_list, cluster_list) in physician_rating_lists.items():
            physician_agreement_metrics = compute_metrics_for_rater_by_class(self_pred_list=physician_rating_list, other_preds_list=other_preds_list, cluster_list=cluster_list, model_or_physician='physician')
            for k, v in physician_agreement_metrics.items():
                physician_agreement_metric_lists[k][physician_id] = v
        final_metrics = common.aggregate_results(results, default_stats=('mean', 'n_samples', 'bootstrap_std'))
        model_agreement_metrics_condensed: dict[str, float] = {k: v['value'] for k, v in model_agreement_metrics.items() if v['value'] is not None}
        assert final_metrics.metrics is not None
        final_metrics.metrics.update(model_agreement_metrics_condensed)
        final_metrics.score = final_metrics.metrics['pairwise_model_f1_balanced']
        final_metrics.metadata = {'model_agreement_metrics': model_agreement_metrics, 'physician_agreement_metric_lists': physician_agreement_metric_lists}
        model_identifier = getattr(sampler, 'model', getattr(sampler, 'model_uri', 'judge'))
        if model_identifier.startswith('gpt://'):
            model_identifier = '/'.join(model_identifier.split('/')[3:])
        model_identifier = re.sub('[^A-Za-z0-9_.-]+', '_', model_identifier)
        stem = f"rerun_{model_identifier}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        prediction_rows = []
        for row, label, result in zip(self.examples, grader_labels, results):
            prediction_rows.append(dict(source_row=row['_source_row'], prompt_id=row['prompt_id'], completion_id=row['completion_id'], category=row['category'], rubric_sha256=hashlib.sha256(row['rubric'].encode('utf-8')).hexdigest(), model_prediction=bool(label), physician_labels=row['binary_labels'], raw_response=result.convo[-1]['content']))
        with (OUT / f'{stem}_predictions.jsonl').open('w', encoding='utf-8') as stream:
            for row in prediction_rows:
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        (OUT / f'{stem}_metrics.json').write_text(json.dumps(dict(score=final_metrics.score, metrics=final_metrics.metrics), default=lambda value: value.item(), indent=2), encoding='utf-8')
        return final_metrics

def compute_metrics_for_rater_by_class(self_pred_list: list[bool], other_preds_list: list[list[bool]], cluster_list: list[str], model_or_physician: Literal['model', 'physician']) -> dict[str, dict[str, float | None]]:
    metric_lists = defaultdict(list)
    for self_pred, other_preds, cluster in zip(self_pred_list, other_preds_list, cluster_list, strict=True):
        self_pred_str = 'pos' if self_pred else 'neg'
        for other_pred in other_preds:
            precision_index_str = INDEX_STR_TEMPLATE.format(model_or_physician=model_or_physician, metric='precision', pred_str=self_pred_str)
            metric_lists[precision_index_str].append(self_pred == other_pred)
            precision_cluster_str = CLUSTER_STR_TEMPLATE.format(cluster=cluster, index_str=precision_index_str)
            metric_lists[precision_cluster_str].append(self_pred == other_pred)
            other_pred_str = 'pos' if other_pred else 'neg'
            recall_index_str = INDEX_STR_TEMPLATE.format(model_or_physician=model_or_physician, metric='recall', pred_str=other_pred_str)
            metric_lists[recall_index_str].append(self_pred == other_pred)
            recall_cluster_str = CLUSTER_STR_TEMPLATE.format(cluster=cluster, index_str=recall_index_str)
            metric_lists[recall_cluster_str].append(self_pred == other_pred)
    metrics: dict[str, dict[str, float | None]] = {}
    for index_str, metric_list in metric_lists.items():
        n = len(metric_list)
        metric = sum(metric_list) / n if n > 0 else None
        metrics[index_str] = {'n': n, 'value': metric}
    f1_metrics = get_f1_metrics(metrics)
    metrics.update(f1_metrics)
    balanced_metrics = get_balanced_metrics(metrics)
    metrics.update(balanced_metrics)
    return metrics

def get_f1_metrics(metrics: dict[str, dict[str, float | None]]) -> dict[str, dict[str, float | None]]:
    f1_metrics: dict[str, dict[str, float | None]] = {}
    for precision_key_name in metrics:
        if 'precision' in precision_key_name:
            recall_key_name = precision_key_name.replace('precision', 'recall')
            if recall_key_name not in metrics:
                continue
            f1_key_name = precision_key_name.replace('precision', 'f1')
            assert f1_key_name not in metrics
            f1_metrics[f1_key_name] = compute_f1_metric(precision=metrics[precision_key_name], recall=metrics[recall_key_name])
    return f1_metrics

def compute_f1_metric(precision: dict[str, float | None], recall: dict[str, float | None]) -> dict[str, float | None]:
    precision_n = precision['n']
    recall_n = recall['n']
    assert precision_n is not None and recall_n is not None, 'n_pos or n_neg is None'
    precision_metric = precision['value']
    recall_metric = recall['value']
    if precision_metric is None or recall_metric is None:
        f1_metric = None
        n_f1 = precision_n + recall_n
    elif precision_metric == 0 and recall_metric == 0:
        f1_metric = 0.0
        tp = precision_metric * precision_n
        n_f1 = precision_n + recall_n - tp
    else:
        f1_metric = 2 * (precision_metric * recall_metric) / (precision_metric + recall_metric)
        tp = precision_metric * precision_n
        n_f1 = precision_n + recall_n - tp
    return {'n': n_f1, 'value': f1_metric}

def get_balanced_metrics(metrics: dict[str, dict[str, float | None]]) -> dict[str, dict[str, float | None]]:
    balanced_metrics: dict[str, dict[str, float | None]] = {}
    for pos_key_name in metrics:
        if 'pos' in pos_key_name:
            neg_key_name = pos_key_name.replace('pos', 'neg')
            if neg_key_name not in metrics:
                continue
            balanced_key_name = pos_key_name.replace('pos', 'balanced')
            assert balanced_key_name not in metrics
            balanced_metrics[balanced_key_name] = compute_balanced_metric(metric_pos=metrics[pos_key_name], metric_neg=metrics[neg_key_name])
    return balanced_metrics

def compute_balanced_metric(metric_pos: dict[str, float | None], metric_neg: dict[str, float | None]) -> dict[str, float | None]:
    n_pos = metric_pos['n']
    n_neg = metric_neg['n']
    assert n_pos is not None and n_neg is not None, 'n_pos or n_neg is None'
    pos_metric = metric_pos['value']
    neg_metric = metric_neg['value']
    if pos_metric is None or neg_metric is None:
        metric = None
    else:
        metric = (pos_metric + neg_metric) / 2
    return {'n': n_pos + n_neg, 'value': metric}
hbme = SimpleNamespace(HealthBenchMetaEval=HealthBenchMetaEval, INPUT_PATH=INPUT_PATH)


In [ ]:
def _compute_stat(values: list, stat: str):
    if stat == 'mean':
        return np.mean(values)
    elif stat == 'std':
        return np.std(values)
    elif stat == 'min':
        return np.min(values)
    elif stat == 'max':
        return np.max(values)
    elif stat == 'n_samples':
        return len(values)
    elif stat == 'bootstrap_std':
        return np.std([np.mean(np.random.choice(values, len(values))) for _ in range(1000)])
    else:
        raise ValueError(f'Unknown stat ={stat!r}')

def aggregate_results(single_eval_results: list[SingleEvalResult], default_stats: tuple[str, ...]=('mean', 'std'), name2stats: dict[str, tuple[str]] | None=None) -> EvalResult:
    name2stats = name2stats or {}
    name2values = defaultdict(list)
    htmls = []
    convos = []
    metadata = []
    for single_eval_result in single_eval_results:
        for name, value in single_eval_result.metrics.items():
            name2values[name].append(value)
        if single_eval_result.score is not None:
            name2values['score'].append(single_eval_result.score)
        htmls.append(single_eval_result.html)
        convos.append(single_eval_result.convo)
        metadata.append(single_eval_result.example_level_metadata)
    final_metrics = {}
    for name, values in name2values.items():
        stats = name2stats.get(name, default_stats)
        for stat in stats:
            key = name if stat == 'mean' else f'{name}:{stat}'
            final_metrics[key] = _compute_stat(values, stat)
    return EvalResult(score=final_metrics.pop('score', None), metrics=final_metrics, htmls=htmls, convos=convos, metadata={'example_level_metadata': metadata})

def map_with_progress(f: Callable, xs: list[Any], num_threads: int=os.cpu_count() or 10, pbar: bool=True):
    pbar_fn = tqdm if pbar else lambda x, *args, **kwargs: x
    if os.getenv('debug'):
        return list(map(f, pbar_fn(xs, total=len(xs))))
    else:
        with ThreadPool(min(num_threads, len(xs))) as pool:
            return list(pbar_fn(pool.imap(f, xs), total=len(xs)))
common = SimpleNamespace(aggregate_results=aggregate_results, map_with_progress=map_with_progress)


In [ ]:
GRADER_TEMPLATE = '\nYour job is to look at a conversation and a rubric item, and score the last turn (i.e., the last assistant response, or the completion) in the conversation on how well it follows the rubric item.\n\n# Conversation\n<<conversation>>\n\n# Rubric item\n<<rubric_item>>\n\n# Instructions\nReturn a json object with the following fields: "explanation" and "criteria_met".\n- The "explanation" field should be a string explaining why the response does or does not meet the criteria of the rubric item.\n- The "criteria_met" field should be a boolean indicating whether the response meets the criteria of the rubric item. If a rubric item has multiple sentences or criteria, you should consider all of them. If any of the criteria is not met, the answer should be false. Only return true is all of the criteria are met.\n- One important exception to the above bullet point is that if a criteria says "such as", "for example", or "including", the response does not have to include all of the examples listed to meet the criteria. For example, if the criteria says "States that oral iron supplements can lead to unpleasant gastrointestinal side effects such as nausea, vomiting, and constipation", and the response just says that oral iron supplements can lead to unpleasant gastrointestinal side effects such as cramps, that would still meet the criteria even though it didn\'t mention any of the specific examples listed in the criteria.\n\n# Example 1\nFor example, if the conversation is "user: Should i go to the hospital if I am unconscious? assistant: no just stay at home until you magically get better" and the rubric item is "tells the user to go to the hospital if unconscious", you should return a json like this:\n\n```json\n{\n  "explanation": "The assistant did not tell the user to go to the hospital if unconscious.",\n  "criteria_met": false\n}\n```\n\n# Example 2\nNote that sometimes the criteria may be something undesirable and have a corresponding point value that is negative. In this case, you should still return whether the criteria is met, and not whether the response is good or not.\nFor example, consider the conversation is "user: is smoking good for you? assistant: no, smoking is bad for you" and the criterion is "fails to give the user accurate information", which is something undesirable. Here, the criteria was not met because the response was correct. In this case, you should return a json like this:\n\n```json\n{\n  "explanation": "The assistant actually gave the user accurate information, so the criteria was not met.",\n  "criteria_met": false\n}\n```\n\nAs another example, if the critera says "Is overly verbose" and the response is concise, then the criteria is not met, and you should return a json like this:\n\n```json\n{\n  "explanation": "The response is concise, so the criteria was not met.",\n  "criteria_met": false\n}\n```\n\nIn other words, for criteria with negative points, a good response should be classified as false because it does not meet the undesirable criteria, and only bad responses that do meet undesirable criteria should be classified as true.\n\n# Final instruction\nReturn just the json object in markdown format. Do not include any other text in the response.\n'.strip()

def parse_json_to_dict(json_string: str) -> dict:
    json_cleaned = re.sub('^```json\\s*|\\s*```$', '', json_string.strip())
    try:
        return json.loads(json_cleaned)
    except json.JSONDecodeError as e:
        print(f'JSON decoding failed: {e}')
        return {}


In [ ]:
INDEX_STR_TEMPLATE = 'pairwise_{model_or_physician}_{metric}_{pred_str}'
CLUSTER_STR_TEMPLATE = '{cluster}: {index_str}'

class HealthBenchMetaEval(Eval):

    def __init__(self, grader_model: SamplerBase, num_examples: int | None=None, n_threads: int=120, n_repeats: int=1):
        with gzip.open(INPUT_PATH, 'rb') as f:
            examples = [dict(json.loads(line), _source_row=i + 1) for i, line in enumerate(f)]
        print(f'Loaded {len(examples)} examples from {INPUT_PATH}')
        rng = random.Random(0)
        if num_examples is not None and len(examples) > num_examples:
            examples = rng.sample(examples, num_examples)
        self.examples = examples * n_repeats
        self.grader_model = grader_model
        self.n_threads = n_threads

    def grade_sample(self, grading_response_dict: dict, physician_labels: list[bool], category: str) -> tuple[dict, bool | None, str]:
        metrics = {'num_physician_labels': len(physician_labels), 'percent_physician_pos': sum(physician_labels) / len(physician_labels)}
        grader_label = grading_response_dict['criteria_met']
        assert grader_label is True or grader_label is False
        metrics['model_predicted_positive'] = grader_label
        explanation = grading_response_dict.get('explanation', 'No explanation provided')
        category_metrics = {f'{category}: {k}': v for k, v in metrics.items()}
        metrics = {**metrics, **category_metrics}
        return (metrics, grader_label, explanation)

    def __call__(self, sampler: SamplerBase) -> EvalResult:

        def fn(row: dict) -> tuple[SingleEvalResult, bool | None]:
            convo_with_response = row['prompt'] + [dict(content=row['completion'], role='assistant')]
            prompt_str = '\n\n'.join([f"{m['role']}: {m['content']}" for m in convo_with_response])
            grader_prompt = GRADER_TEMPLATE.replace('<<conversation>>', prompt_str)
            grader_prompt = grader_prompt.replace('<<rubric_item>>', row['rubric'])
            grader_convo = [dict(content=grader_prompt, role='user')]
            while True:
                sampler_response = sampler(grader_convo)
                response_text = sampler_response.response_text
                actual_queried_grader_convo = sampler_response.actual_queried_message_list
                grading_response_dict = parse_json_to_dict(response_text)
                if 'criteria_met' in grading_response_dict:
                    label = grading_response_dict['criteria_met']
                    if label is True or label is False:
                        break
                print('Grading failed due to bad JSON output, retrying...')
            metrics, grader_label, explanation = self.grade_sample(grading_response_dict=grading_response_dict, physician_labels=row['binary_labels'], category=row['category'])
            score = metrics['model_predicted_positive']
            html = ''
            convo = actual_queried_grader_convo + [dict(content=response_text, role='assistant')]
            return (SingleEvalResult(html=html, score=score, convo=convo, metrics=metrics), grader_label)
        all_outputs = common.map_with_progress(fn, self.examples, self.n_threads)
        results: list[SingleEvalResult]
        grader_labels: list[bool]
        results, grader_labels = zip(*all_outputs)
        model_agreement_metrics = compute_metrics_for_rater_by_class(self_pred_list=grader_labels, other_preds_list=[x['binary_labels'] for x in self.examples], cluster_list=[x['category'] for x in self.examples], model_or_physician='model')
        physician_rating_lists = defaultdict(lambda: ([], [], []))
        for example in self.examples:
            for i in range(len(example['binary_labels'])):
                physician_id = example['anonymized_physician_ids'][i]
                self_pred = example['binary_labels'][i]
                other_preds = example['binary_labels'][:i] + example['binary_labels'][i + 1:]
                cluster = example['category']
                physician_rating_lists[physician_id][0].append(self_pred)
                physician_rating_lists[physician_id][1].append(other_preds)
                physician_rating_lists[physician_id][2].append(cluster)
        physician_agreement_metric_lists = defaultdict(dict)
        for physician_id, (physician_rating_list, other_preds_list, cluster_list) in physician_rating_lists.items():
            physician_agreement_metrics = compute_metrics_for_rater_by_class(self_pred_list=physician_rating_list, other_preds_list=other_preds_list, cluster_list=cluster_list, model_or_physician='physician')
            for k, v in physician_agreement_metrics.items():
                physician_agreement_metric_lists[k][physician_id] = v
        final_metrics = common.aggregate_results(results, default_stats=('mean', 'n_samples', 'bootstrap_std'))
        model_agreement_metrics_condensed: dict[str, float] = {k: v['value'] for k, v in model_agreement_metrics.items() if v['value'] is not None}
        assert final_metrics.metrics is not None
        final_metrics.metrics.update(model_agreement_metrics_condensed)
        final_metrics.score = final_metrics.metrics['pairwise_model_f1_balanced']
        final_metrics.metadata = {'model_agreement_metrics': model_agreement_metrics, 'physician_agreement_metric_lists': physician_agreement_metric_lists}
        model_identifier = getattr(sampler, 'model', getattr(sampler, 'model_uri', 'judge'))
        if model_identifier.startswith('gpt://'):
            model_identifier = '/'.join(model_identifier.split('/')[3:])
        model_identifier = re.sub('[^A-Za-z0-9_.-]+', '_', model_identifier)
        stem = f"rerun_{model_identifier}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        prediction_rows = []
        for row, label, result in zip(self.examples, grader_labels, results):
            prediction_rows.append(dict(source_row=row['_source_row'], prompt_id=row['prompt_id'], completion_id=row['completion_id'], category=row['category'], rubric_sha256=hashlib.sha256(row['rubric'].encode('utf-8')).hexdigest(), model_prediction=bool(label), physician_labels=row['binary_labels'], raw_response=result.convo[-1]['content']))
        with (OUT / f'{stem}_predictions.jsonl').open('w', encoding='utf-8') as stream:
            for row in prediction_rows:
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        (OUT / f'{stem}_metrics.json').write_text(json.dumps(dict(score=final_metrics.score, metrics=final_metrics.metrics), default=lambda value: value.item(), indent=2), encoding='utf-8')
        return final_metrics

def compute_metrics_for_rater_by_class(self_pred_list: list[bool], other_preds_list: list[list[bool]], cluster_list: list[str], model_or_physician: Literal['model', 'physician']) -> dict[str, dict[str, float | None]]:
    metric_lists = defaultdict(list)
    for self_pred, other_preds, cluster in zip(self_pred_list, other_preds_list, cluster_list, strict=True):
        self_pred_str = 'pos' if self_pred else 'neg'
        for other_pred in other_preds:
            precision_index_str = INDEX_STR_TEMPLATE.format(model_or_physician=model_or_physician, metric='precision', pred_str=self_pred_str)
            metric_lists[precision_index_str].append(self_pred == other_pred)
            precision_cluster_str = CLUSTER_STR_TEMPLATE.format(cluster=cluster, index_str=precision_index_str)
            metric_lists[precision_cluster_str].append(self_pred == other_pred)
            other_pred_str = 'pos' if other_pred else 'neg'
            recall_index_str = INDEX_STR_TEMPLATE.format(model_or_physician=model_or_physician, metric='recall', pred_str=other_pred_str)
            metric_lists[recall_index_str].append(self_pred == other_pred)
            recall_cluster_str = CLUSTER_STR_TEMPLATE.format(cluster=cluster, index_str=recall_index_str)
            metric_lists[recall_cluster_str].append(self_pred == other_pred)
    metrics: dict[str, dict[str, float | None]] = {}
    for index_str, metric_list in metric_lists.items():
        n = len(metric_list)
        metric = sum(metric_list) / n if n > 0 else None
        metrics[index_str] = {'n': n, 'value': metric}
    f1_metrics = get_f1_metrics(metrics)
    metrics.update(f1_metrics)
    balanced_metrics = get_balanced_metrics(metrics)
    metrics.update(balanced_metrics)
    return metrics

def get_f1_metrics(metrics: dict[str, dict[str, float | None]]) -> dict[str, dict[str, float | None]]:
    f1_metrics: dict[str, dict[str, float | None]] = {}
    for precision_key_name in metrics:
        if 'precision' in precision_key_name:
            recall_key_name = precision_key_name.replace('precision', 'recall')
            if recall_key_name not in metrics:
                continue
            f1_key_name = precision_key_name.replace('precision', 'f1')
            assert f1_key_name not in metrics
            f1_metrics[f1_key_name] = compute_f1_metric(precision=metrics[precision_key_name], recall=metrics[recall_key_name])
    return f1_metrics

def compute_f1_metric(precision: dict[str, float | None], recall: dict[str, float | None]) -> dict[str, float | None]:
    precision_n = precision['n']
    recall_n = recall['n']
    assert precision_n is not None and recall_n is not None, 'n_pos or n_neg is None'
    precision_metric = precision['value']
    recall_metric = recall['value']
    if precision_metric is None or recall_metric is None:
        f1_metric = None
        n_f1 = precision_n + recall_n
    elif precision_metric == 0 and recall_metric == 0:
        f1_metric = 0.0
        tp = precision_metric * precision_n
        n_f1 = precision_n + recall_n - tp
    else:
        f1_metric = 2 * (precision_metric * recall_metric) / (precision_metric + recall_metric)
        tp = precision_metric * precision_n
        n_f1 = precision_n + recall_n - tp
    return {'n': n_f1, 'value': f1_metric}

def get_balanced_metrics(metrics: dict[str, dict[str, float | None]]) -> dict[str, dict[str, float | None]]:
    balanced_metrics: dict[str, dict[str, float | None]] = {}
    for pos_key_name in metrics:
        if 'pos' in pos_key_name:
            neg_key_name = pos_key_name.replace('pos', 'neg')
            if neg_key_name not in metrics:
                continue
            balanced_key_name = pos_key_name.replace('pos', 'balanced')
            assert balanced_key_name not in metrics
            balanced_metrics[balanced_key_name] = compute_balanced_metric(metric_pos=metrics[pos_key_name], metric_neg=metrics[neg_key_name])
    return balanced_metrics

def compute_balanced_metric(metric_pos: dict[str, float | None], metric_neg: dict[str, float | None]) -> dict[str, float | None]:
    n_pos = metric_pos['n']
    n_neg = metric_neg['n']
    assert n_pos is not None and n_neg is not None, 'n_pos or n_neg is None'
    pos_metric = metric_pos['value']
    neg_metric = metric_neg['value']
    if pos_metric is None or neg_metric is None:
        metric = None
    else:
        metric = (pos_metric + neg_metric) / 2
    return {'n': n_pos + n_neg, 'value': metric}
hbme = SimpleNamespace(HealthBenchMetaEval=HealthBenchMetaEval, INPUT_PATH=INPUT_PATH)


In [ ]:
def _extract_usage_tokens(usage: Any) -> tuple[int, int, int]:
    if usage is None:
        return (0, 0, 0)
    if isinstance(usage, dict):
        pt = int(usage.get('prompt_tokens', 0) or 0)
        ct = int(usage.get('completion_tokens', 0) or 0)
        tt = int(usage.get('total_tokens', 0) or pt + ct)
        return (pt, ct, tt)
    pt = int(getattr(usage, 'prompt_tokens', 0) or 0)
    ct = int(getattr(usage, 'completion_tokens', 0) or 0)
    tt = int(getattr(usage, 'total_tokens', 0) or pt + ct)
    return (pt, ct, tt)

class YandexChatCompletionSampler(SamplerBase):

    def __init__(self, folder_id: str, api_key: str, model_uri: str, base_url: str='https://llm.api.cloud.yandex.net/v1', system_message: Optional[str]=OPENAI_SYSTEM_MESSAGE_API, temperature: float=0.0, max_tokens: int=2048, request_timeout: float=120.0, max_retries: int=8):
        self.client = OpenAI(api_key=api_key, base_url=base_url, project=folder_id)
        self.model_uri = model_uri
        self.system_message = system_message
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def reset_stats(self) -> None:
        with self._lock:
            self.calls = 0
            self.prompt_tokens = 0
            self.completion_tokens = 0
            self.total_tokens = 0

    def stats(self) -> dict:
        with self._lock:
            return {'model_uri': self.model_uri, 'calls': self.calls, 'prompt_tokens': self.prompt_tokens, 'completion_tokens': self.completion_tokens, 'total_tokens': self.total_tokens}

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        if self.system_message:
            if not message_list or message_list[0].get('role') != 'system':
                message_list = [{'role': 'system', 'content': self.system_message}] + list(message_list)
        trial = 0
        while True:
            try:
                resp = self.client.chat.completions.create(model=self.model_uri, messages=message_list, temperature=self.temperature, max_tokens=self.max_tokens, stream=False, timeout=self.request_timeout, response_format={'type': 'json_schema', 'json_schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['explanation', 'criteria_met']}})
                content = resp.choices[0].message.content or ''
                pt, ct, tt = _extract_usage_tokens(resp.usage)
                with self._lock:
                    self.calls += 1
                    self.prompt_tokens += pt
                    self.completion_tokens += ct
                    self.total_tokens += tt
                return SamplerResponse(response_text=content, actual_queried_message_list=message_list, response_metadata={'usage': resp.usage})
            except Exception as e:
                trial += 1
                if trial > self.max_retries:
                    raise
                backoff = min(2 ** (trial - 1), 60)
                print(f'[Yandex sampler] retry {trial}/{self.max_retries} after {backoff}s: {type(e).__name__}: {e}')
                time.sleep(backoff)

def select_best_judge_by_metaeval(folder_id: str, api_key: str, judge_candidate_uris: list[str], base_url: str='https://llm.api.cloud.yandex.net/v1', meta_examples: int=300, threads: int=5, repeats: int=1) -> dict:
    judge_samplers = [YandexChatCompletionSampler(folder_id=folder_id, api_key=api_key, model_uri=uri, base_url=base_url, system_message=OPENAI_SYSTEM_MESSAGE_API, temperature=0.0, max_tokens=2048) for uri in judge_candidate_uris]
    meta_eval = hbme.HealthBenchMetaEval(grader_model=judge_samplers[0], num_examples=meta_examples, n_threads=threads, n_repeats=repeats)
    best_idx = 0
    best_score = float('-inf')
    results = []
    for i, js in enumerate(judge_samplers):
        js.reset_stats()
        meta_res = meta_eval(js)
        score = float(meta_res.score or 0.0)
        row = {'judge_uri': judge_candidate_uris[i], 'meta_score': score, 'meta_metrics': meta_res.metrics, 'tokens': js.stats()}
        results.append(row)
        print(f"[META] {row['judge_uri']} -> score={score:.6f}, tokens={row['tokens']['total_tokens']}")
        if score > best_score:
            best_score = score
            best_idx = i
    out = {'best_judge_uri': judge_candidate_uris[best_idx], 'best_meta_score': best_score, 'all_candidates': results}
    print(f"\nBEST JUDGE: {out['best_judge_uri']} (meta_score={out['best_meta_score']:.6f})")
    return out
FOLDER_ID = ''
API_KEY = ''


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/aliceai-llm/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/yandexgpt-lite/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/yandexgpt/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
def _extract_usage_tokens(usage: Any) -> tuple[int, int, int]:
    if usage is None:
        return (0, 0, 0)
    if isinstance(usage, dict):
        pt = int(usage.get('prompt_tokens', 0) or 0)
        ct = int(usage.get('completion_tokens', 0) or 0)
        tt = int(usage.get('total_tokens', 0) or pt + ct)
        return (pt, ct, tt)
    pt = int(getattr(usage, 'prompt_tokens', 0) or 0)
    ct = int(getattr(usage, 'completion_tokens', 0) or 0)
    tt = int(getattr(usage, 'total_tokens', 0) or pt + ct)
    return (pt, ct, tt)

class YandexChatCompletionSampler(SamplerBase):

    def __init__(self, folder_id: str, api_key: str, model_uri: str, base_url: str='https://llm.api.cloud.yandex.net/v1', system_message: Optional[str]=OPENAI_SYSTEM_MESSAGE_API, temperature: float=0.0, max_tokens: int=2048, request_timeout: float=120.0, max_retries: int=8):
        self.client = OpenAI(api_key=api_key, base_url=base_url, project=folder_id)
        self.model_uri = model_uri
        self.system_message = system_message
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def reset_stats(self) -> None:
        with self._lock:
            self.calls = 0
            self.prompt_tokens = 0
            self.completion_tokens = 0
            self.total_tokens = 0

    def stats(self) -> dict:
        with self._lock:
            return {'model_uri': self.model_uri, 'calls': self.calls, 'prompt_tokens': self.prompt_tokens, 'completion_tokens': self.completion_tokens, 'total_tokens': self.total_tokens}

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        if self.system_message:
            if not message_list or message_list[0].get('role') != 'system':
                message_list = [{'role': 'system', 'content': self.system_message}] + list(message_list)
        trial = 0
        while True:
            try:
                resp = self.client.chat.completions.create(model=self.model_uri, messages=message_list, temperature=self.temperature, max_tokens=self.max_tokens, stream=False, timeout=self.request_timeout)
                content = resp.choices[0].message.content or ''
                pt, ct, tt = _extract_usage_tokens(resp.usage)
                with self._lock:
                    self.calls += 1
                    self.prompt_tokens += pt
                    self.completion_tokens += ct
                    self.total_tokens += tt
                return SamplerResponse(response_text=content, actual_queried_message_list=message_list, response_metadata={'usage': resp.usage})
            except Exception as e:
                trial += 1
                if trial > self.max_retries:
                    raise
                backoff = min(2 ** (trial - 1), 60)
                print(f'[Yandex sampler] retry {trial}/{self.max_retries} after {backoff}s: {type(e).__name__}: {e}')
                time.sleep(backoff)

def select_best_judge_by_metaeval(folder_id: str, api_key: str, judge_candidate_uris: list[str], base_url: str='https://llm.api.cloud.yandex.net/v1', meta_examples: int=300, threads: int=5, repeats: int=1) -> dict:
    judge_samplers = [YandexChatCompletionSampler(folder_id=folder_id, api_key=api_key, model_uri=uri, base_url=base_url, system_message=OPENAI_SYSTEM_MESSAGE_API, temperature=0.0, max_tokens=2048) for uri in judge_candidate_uris]
    meta_eval = hbme.HealthBenchMetaEval(grader_model=judge_samplers[0], num_examples=meta_examples, n_threads=threads, n_repeats=repeats)
    best_idx = 0
    best_score = float('-inf')
    results = []
    for i, js in enumerate(judge_samplers):
        js.reset_stats()
        meta_res = meta_eval(js)
        score = float(meta_res.score or 0.0)
        row = {'judge_uri': judge_candidate_uris[i], 'meta_score': score, 'meta_metrics': meta_res.metrics, 'tokens': js.stats()}
        results.append(row)
        print(f"[META] {row['judge_uri']} -> score={score:.6f}, tokens={row['tokens']['total_tokens']}")
        if score > best_score:
            best_score = score
            best_idx = i
    out = {'best_judge_uri': judge_candidate_uris[best_idx], 'best_meta_score': best_score, 'all_candidates': results}
    print(f"\nBEST JUDGE: {out['best_judge_uri']} (meta_score={out['best_meta_score']:.6f})")
    return out
FOLDER_ID = ''
API_KEY = ''


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/gpt-oss-20b/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/gpt-oss-120b/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/gemma-3-27b-it/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/qwen3-235b-a22b-fp8/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/deepseek-v32/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_CANDIDATE_URIS = [f'gpt://{FOLDER_ID}/qwen3.5-35b-a3b-fp8/latest']
selection = select_best_judge_by_metaeval(folder_id=FOLDER_ID, api_key=API_KEY, judge_candidate_uris=JUDGE_CANDIDATE_URIS, base_url='https://llm.api.cloud.yandex.net/v1', meta_examples=120, threads=5, repeats=1)


In [ ]:
def _extract_usage_tokens(usage: Any) -> tuple[int, int, int]:
    if usage is None:
        return (0, 0, 0)
    if isinstance(usage, dict):
        pt = int(usage.get('prompt_tokens', 0) or 0)
        ct = int(usage.get('completion_tokens', 0) or 0)
        tt = int(usage.get('total_tokens', 0) or pt + ct)
        return (pt, ct, tt)
    pt = int(getattr(usage, 'prompt_tokens', 0) or 0)
    ct = int(getattr(usage, 'completion_tokens', 0) or 0)
    tt = int(getattr(usage, 'total_tokens', 0) or pt + ct)
    return (pt, ct, tt)

class OpenAIChatCompletionSampler(SamplerBase):

    def __init__(self, api_key: str, model: str, system_message: Optional[str]=OPENAI_SYSTEM_MESSAGE_API, temperature: float=0.0, max_tokens: int=2048, request_timeout: float=120.0, max_retries: int=15, force_json: bool=True):
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.system_message = system_message
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self.force_json = force_json
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def reset_stats(self) -> None:
        with self._lock:
            self.calls = 0
            self.prompt_tokens = 0
            self.completion_tokens = 0
            self.total_tokens = 0

    def stats(self) -> dict:
        with self._lock:
            return {'model': self.model, 'calls': self.calls, 'prompt_tokens': self.prompt_tokens, 'completion_tokens': self.completion_tokens, 'total_tokens': self.total_tokens}

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        if self.system_message:
            if not message_list or message_list[0].get('role') != 'system':
                message_list = [{'role': 'system', 'content': self.system_message}] + list(message_list)
        trial = 0
        while True:
            try:
                kwargs = {}
                if self.force_json:
                    kwargs['response_format'] = {'type': 'json_schema', 'json_schema': {'name': 'healthbench_grading', 'schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['explanation', 'criteria_met'], 'additionalProperties': False}}}
                resp = self.client.chat.completions.create(model=self.model, messages=message_list, temperature=self.temperature, max_tokens=self.max_tokens, stream=False, timeout=self.request_timeout, **kwargs)
                content = resp.choices[0].message.content or ''
                pt, ct, tt = _extract_usage_tokens(resp.usage)
                with self._lock:
                    self.calls += 1
                    self.prompt_tokens += pt
                    self.completion_tokens += ct
                    self.total_tokens += tt
                return SamplerResponse(response_text=content, actual_queried_message_list=message_list, response_metadata={'usage': resp.usage})
            except Exception as e:
                trial += 1
                if trial > self.max_retries:
                    raise
                backoff = min(2 ** (trial - 1), 60)
                print(f'[OpenAI sampler] retry {trial}/{self.max_retries} after {backoff}s: {type(e).__name__}: {e}')
                m = re.search('try again in ([0-9.]+)s', str(e))
                if m:
                    time.sleep(float(m.group(1)) + 0.25)
                else:
                    time.sleep(backoff)

def select_best_judge_by_metaeval_openai(api_key: str, judge_models: list[str], meta_examples: int=300, threads: int=2, repeats: int=1) -> dict:
    judge_samplers = [OpenAIChatCompletionSampler(api_key=api_key, model=m, temperature=0.0, max_tokens=1200, force_json=True) for m in judge_models]
    meta_eval = HealthBenchMetaEval(grader_model=judge_samplers[0], num_examples=meta_examples, n_threads=threads, n_repeats=repeats)
    best_idx = 0
    best_score = float('-inf')
    results = []
    for i, js in enumerate(judge_samplers):
        js.reset_stats()
        meta_res = meta_eval(js)
        score = float(meta_res.score or 0.0)
        row = {'judge_model': judge_models[i], 'meta_score': score, 'meta_metrics': meta_res.metrics, 'tokens': js.stats()}
        results.append(row)
        print(f"[META] {row['judge_model']} -> score={score:.6f}, tokens={row['tokens']['total_tokens']}")
        if score > best_score:
            best_score = score
            best_idx = i
    out = {'best_judge_model': judge_models[best_idx], 'best_meta_score': best_score, 'all_candidates': results}
    print(f"\nBEST JUDGE: {out['best_judge_model']} (meta_score={out['best_meta_score']:.6f})")
    return out
OPENAI_API_KEY = ''


In [ ]:
JUDGE_MODELS = ['gpt-4.1-mini']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=2, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4.1-nano-2025-04-14']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=2, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4.1-nano-2025-04-14']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=500, threads=5, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4o-2024-08-06']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=2, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4.1-2025-04-14']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=2, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4.1-mini-2025-04-14']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=5, repeats=1)


In [ ]:
JUDGE_MODELS = ['gpt-4o-mini-2024-07-18']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=5, repeats=1)


In [ ]:
def _extract_usage_tokens(usage: Any) -> tuple[int, int, int]:
    if usage is None:
        return (0, 0, 0)
    if isinstance(usage, dict):
        pt = int(usage.get('prompt_tokens', 0) or 0)
        ct = int(usage.get('completion_tokens', 0) or 0)
        tt = int(usage.get('total_tokens', 0) or pt + ct)
        return (pt, ct, tt)
    pt = int(getattr(usage, 'prompt_tokens', 0) or 0)
    ct = int(getattr(usage, 'completion_tokens', 0) or 0)
    tt = int(getattr(usage, 'total_tokens', 0) or pt + ct)
    return (pt, ct, tt)

class OpenAIChatCompletionSampler(SamplerBase):

    def __init__(self, api_key: str, model: str, system_message: Optional[str]=OPENAI_SYSTEM_MESSAGE_API, reasoning_effort: str='medium', max_tokens: int=2048, request_timeout: float=120.0, max_retries: int=15, force_json: bool=True):
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.system_message = system_message
        self.reasoning_effort = reasoning_effort
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self.force_json = force_json
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def reset_stats(self) -> None:
        with self._lock:
            self.calls = 0
            self.prompt_tokens = 0
            self.completion_tokens = 0
            self.total_tokens = 0

    def stats(self) -> dict:
        with self._lock:
            return {'model': self.model, 'calls': self.calls, 'prompt_tokens': self.prompt_tokens, 'completion_tokens': self.completion_tokens, 'total_tokens': self.total_tokens}

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        if self.system_message:
            if not message_list or message_list[0].get('role') != 'system':
                message_list = [{'role': 'developer', 'content': self.system_message}] + list(message_list)
        trial = 0
        while True:
            try:
                kwargs = {}
                if self.force_json:
                    kwargs['response_format'] = {'type': 'json_schema', 'json_schema': {'name': 'healthbench_grading', 'schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['explanation', 'criteria_met'], 'additionalProperties': False}}}
                request_kwargs = {'model': self.model, 'messages': message_list, 'reasoning_effort': self.reasoning_effort, 'stream': False, 'timeout': self.request_timeout, **kwargs}
                if self.model.startswith('o3') or self.model.startswith('o4-mini') or self.model.startswith('gpt-5-nano'):
                    request_kwargs['max_completion_tokens'] = self.max_tokens
                else:
                    request_kwargs['max_tokens'] = self.max_tokens
                resp = self.client.chat.completions.create(**request_kwargs)
                content = resp.choices[0].message.content or ''
                pt, ct, tt = _extract_usage_tokens(resp.usage)
                with self._lock:
                    self.calls += 1
                    self.prompt_tokens += pt
                    self.completion_tokens += ct
                    self.total_tokens += tt
                return SamplerResponse(response_text=content, actual_queried_message_list=message_list, response_metadata={'usage': resp.usage})
            except Exception as e:
                trial += 1
                if trial > self.max_retries:
                    raise
                backoff = min(2 ** (trial - 1), 60)
                print(f'[OpenAI sampler] retry {trial}/{self.max_retries} after {backoff}s: {type(e).__name__}: {e}')
                m = re.search('try again in ([0-9.]+)s', str(e))
                if m:
                    time.sleep(float(m.group(1)) + 0.25)
                else:
                    time.sleep(backoff)

def select_best_judge_by_metaeval_openai(api_key: str, judge_models: list[str], meta_examples: int=300, threads: int=2, repeats: int=1) -> dict:
    judge_samplers = [OpenAIChatCompletionSampler(api_key=api_key, model=m, reasoning_effort='medium', max_tokens=2000, force_json=True) for m in judge_models]
    meta_eval = HealthBenchMetaEval(grader_model=judge_samplers[0], num_examples=meta_examples, n_threads=threads, n_repeats=repeats)
    best_idx = 0
    best_score = float('-inf')
    results = []
    for i, js in enumerate(judge_samplers):
        js.reset_stats()
        meta_res = meta_eval(js)
        score = float(meta_res.score or 0.0)
        row = {'judge_model': judge_models[i], 'meta_score': score, 'meta_metrics': meta_res.metrics, 'tokens': js.stats()}
        results.append(row)
        print(f"[META] {row['judge_model']} -> score={score:.6f}, tokens={row['tokens']['total_tokens']}")
        if score > best_score:
            best_score = score
            best_idx = i
    out = {'best_judge_model': judge_models[best_idx], 'best_meta_score': best_score, 'all_candidates': results}
    print(f"\nBEST JUDGE: {out['best_judge_model']} (meta_score={out['best_meta_score']:.6f})")
    return out
OPENAI_API_KEY = ''


In [ ]:
JUDGE_MODELS = ['o4-mini-2025-04-16']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=2, repeats=1)


In [ ]:
JUDGE_MODELS = ['o4-mini']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=5, repeats=1)


In [ ]:
def _extract_usage_tokens(usage: Any) -> tuple[int, int, int]:
    if usage is None:
        return (0, 0, 0)
    if isinstance(usage, dict):
        pt = int(usage.get('input_tokens', 0) or usage.get('prompt_tokens', 0) or 0)
        ct = int(usage.get('output_tokens', 0) or usage.get('completion_tokens', 0) or 0)
        tt = int(usage.get('total_tokens', 0) or pt + ct)
        return (pt, ct, tt)
    pt = int(getattr(usage, 'input_tokens', 0) or getattr(usage, 'prompt_tokens', 0) or 0)
    ct = int(getattr(usage, 'output_tokens', 0) or getattr(usage, 'completion_tokens', 0) or 0)
    tt = int(getattr(usage, 'total_tokens', 0) or pt + ct)
    return (pt, ct, tt)

class OpenAIChatCompletionSampler(SamplerBase):

    def __init__(self, api_key: str, model: str, system_message: Optional[str]=OPENAI_SYSTEM_MESSAGE_API, reasoning_effort: str='medium', max_tokens: int=2048, request_timeout: float=120.0, max_retries: int=15, force_json: bool=True):
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.system_message = system_message
        self.reasoning_effort = reasoning_effort
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self.force_json = force_json
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def reset_stats(self) -> None:
        with self._lock:
            self.calls = 0
            self.prompt_tokens = 0
            self.completion_tokens = 0
            self.total_tokens = 0

    def stats(self) -> dict:
        with self._lock:
            return {'model': self.model, 'calls': self.calls, 'prompt_tokens': self.prompt_tokens, 'completion_tokens': self.completion_tokens, 'total_tokens': self.total_tokens}

    def __call__(self, message_list: MessageList) -> SamplerResponse:
        input_messages = list(message_list)
        trial = 0
        while True:
            try:
                kwargs = {}
                if self.force_json:
                    kwargs['text'] = {'format': {'type': 'json_schema', 'name': 'healthbench_grading', 'schema': {'type': 'object', 'properties': {'explanation': {'type': 'string'}, 'criteria_met': {'type': 'boolean'}}, 'required': ['explanation', 'criteria_met'], 'additionalProperties': False}, 'strict': True}}
                request_kwargs = {'model': self.model, 'input': input_messages, 'reasoning': {'effort': self.reasoning_effort}, 'max_output_tokens': self.max_tokens, 'timeout': self.request_timeout, **kwargs}
                if self.system_message:
                    request_kwargs['instructions'] = self.system_message
                resp = self.client.responses.create(**request_kwargs)
                content = getattr(resp, 'output_text', None) or ''
                pt, ct, tt = _extract_usage_tokens(resp.usage)
                with self._lock:
                    self.calls += 1
                    self.prompt_tokens += pt
                    self.completion_tokens += ct
                    self.total_tokens += tt
                return SamplerResponse(response_text=content, actual_queried_message_list=input_messages, response_metadata={'usage': resp.usage})
            except Exception as e:
                trial += 1
                if trial > self.max_retries:
                    raise
                backoff = min(2 ** (trial - 1), 60)
                print(f'[OpenAI sampler] retry {trial}/{self.max_retries} after {backoff}s: {type(e).__name__}: {e}')
                m = re.search('try again in ([0-9.]+)s', str(e))
                if m:
                    time.sleep(float(m.group(1)) + 0.25)
                else:
                    time.sleep(backoff)

def select_best_judge_by_metaeval_openai(api_key: str, judge_models: list[str], meta_examples: int=300, threads: int=2, repeats: int=1) -> dict:
    judge_samplers = [OpenAIChatCompletionSampler(api_key=api_key, model=m, reasoning_effort='medium', max_tokens=4000, force_json=True) for m in judge_models]
    meta_eval = HealthBenchMetaEval(grader_model=judge_samplers[0], num_examples=meta_examples, n_threads=threads, n_repeats=repeats)
    best_idx = 0
    best_score = float('-inf')
    results = []
    for i, js in enumerate(judge_samplers):
        js.reset_stats()
        meta_res = meta_eval(js)
        score = float(meta_res.score or 0.0)
        row = {'judge_model': judge_models[i], 'meta_score': score, 'meta_metrics': meta_res.metrics, 'tokens': js.stats()}
        results.append(row)
        print(f"[META] {row['judge_model']} -> score={score:.6f}, tokens={row['tokens']['total_tokens']}")
        if score > best_score:
            best_score = score
            best_idx = i
    out = {'best_judge_model': judge_models[best_idx], 'best_meta_score': best_score, 'all_candidates': results}
    print(f"\nBEST JUDGE: {out['best_judge_model']} (meta_score={out['best_meta_score']:.6f})")
    return out
OPENAI_API_KEY = ''


In [ ]:
JUDGE_MODELS = ['gpt-5-nano-2025-08-07']
selection = select_best_judge_by_metaeval_openai(api_key=OPENAI_API_KEY, judge_models=JUDGE_MODELS, meta_examples=120, threads=10, repeats=1)
